In [ ]:
!pip install spacy
!python -m spacy download en_core_web_lg



In [2]:
import os
import json
import spacy
# import nltk
import requests
import time
import pandas as pd
from spacy import displacy
from nltk.tag import pos_tag
from bs4 import BeautifulSoup as bs
from nltk.tokenize import word_tokenize
# from nltk.sentiment.vader import SentimentIntensityAnalyzer
import csv
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')
nltk.download('popular')


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to
[nltk_data]    |     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]    |   Package cmudict is already up-to-date!
[nltk_data]    | Downloading package gazetteers to
[nltk_data]    |     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]    |   Package gazetteers is already up-to-date!
[nltk_data]    | Downloading package genesis to
[nltk_data]    |     C:

True

In [2]:
def push_news(news_payload):
    newsUrl = "http://103.233.79.196:8083/background_scheduler/news/processNews"
    header = {
        "Content-Type": "application/json"
    }

    resp = requests.post(url = newsUrl, data = json.dumps(news_payload), headers = header)
    print(resp.content)

In [3]:
# nltk.download('averaged_perceptron_tagger')

In [8]:
class NewsData():
    global comp_cins, csv_file_path
    
    
    LABOUR_NEWS_TAG = ["Labour Strikes", "Strikes", "Quits", "unrest",
                       "agitation", "lock out", "termination", "ESI", "unrest", "Fired", "Quit", "suicide", "protest", "injury",
                       "injured", "salary", "EPF", "attrition", "Dispute", "harassment", "protests", "Strike"]

    RAID_NEWS_TAG = ["Raid", "Raids", "Raided"]

    RESIGNATION_NEWS_TAG = ["Resign", "Resigns", "Resigned",
                            "Resignation", "resigning", "management changes", "terminate"]

    FRAUD_REGULATORY_ACTION_NEWS_TAG = ["Fraud", "Embezzlement",
                                         "Money laundering", "Syphoning", "CBI inquiry", "Bribery", "Political Influence", "SEBI proceeding",
                                         "regulatory action", "Penalties", "Fines", "Penalty", "Cheating", "NPA", "Default", "Criminal", "Breach",
                                         "Scam", "Arrest", "Warrant Issued", "Probe", "Fined", "Misappropriation", "Inspection", "Complaint against",
                                         "insolvency", "I T notice", "Hawala", "CBI", "FIR", "Jail", "Illeagal", "Forging", "booked", "scrutinize",
                                         "land grab", "RBI", "CBI", "SEBI", "IRDA", "EXIM", "Fraudulent", "theft", "theif", "bankruptcy",
                                         "Black Money", "Bribe", "Central Bureau of Investigation", "charge sheet", "chargesheet", "cheat",
                                         "corrupt", "corruption", "custody", "Due", "Dues", "evading", "Fine", "Fined", "forgery", "illegal",
                                         "Inspect", "Inspector", "investigation", "Labour Dispute", "land grab", "non compliance", "notices",
                                         "Offered Money", "Penalized", "police", "Serious Fraud", "SFIO", "violation", "Whistleblower"]

    MARKET_STOCK_MOVEMENT_NEWS_TAG = ["plunges", "Slowdown", "Delay",
                                       "Delayed", "downgrade", "downgraded", "losses", "loss", "SEBI", "profit decline", "Wind Up", "plummets",
                                       "Decline", "exits", "low", "pledge", "plummet", "SELL Recommendation", "stake sale", "stress", "suspends",
                                       "worst", "sell", "fall", "down", "drop", "falls", "sells"]

    LITIGATION_NEWS_TAG = ["Case Registered", "Court Denies",
                           "Court Rejects", "contempt", "Registered Case", "Settlement", "allegations", "High Court", "District Court",
                           "Lower Court", "Supreme Court", "Judgement", "copyrights", "defamation", "DRT", "NCLT"]

    WHISTLEBLOWER_NEWS_TAG = ["Whistleblower"]

    ACCIDENT_NEWS_TAG = ["Accident", "Fire", "Death", "Died",
                         "Passed Away", "Die", "casualty", "mishap", "Blast", "casualties", "Died", "killed", "kills", "passes away",
                         "shut down"]


    # Combine all lists into a single list
    all_tags = (LABOUR_NEWS_TAG + RAID_NEWS_TAG + RESIGNATION_NEWS_TAG + FRAUD_REGULATORY_ACTION_NEWS_TAG +
                MARKET_STOCK_MOVEMENT_NEWS_TAG + LITIGATION_NEWS_TAG + WHISTLEBLOWER_NEWS_TAG + ACCIDENT_NEWS_TAG)

    # Print the combined list
    #print(all_tags)

    # load cins of companies
    comp_cins = pd.read_excel(r'D:\Nexensus_Projects\News\company_cin_symbol.xlsx')
    # Load News model
    NER_sm = spacy.load("en_core_web_sm")
    NER = spacy.load("en_core_web_lg")
    
    # # Load CSV file into a DataFrame
    # csv_file_path = r'I:\NewsData\SentimentalWords.csv'  # Replace with the actual path
    # df_csv = pd.read_csv(csv_file_path)

    # # Map sentiment labels to numerical values
    # sentiment_mapping = {'Negative': -1, 'Neutral': 0, 'Positive': 1}

    # # Create a dictionary mapping words to sentiments from CSV
    # word_sentiments_csv = dict(zip(df_csv['text'], df_csv['sentiment'].map(sentiment_mapping)))

    # # Initialize the Sentiment Intensity Analyzer
    # sia = SentimentIntensityAnalyzer()

    # def get_sentence_sentiment(self, sentence):
    #     # Tokenize the sentence
    #     words = word_tokenize(sentence.lower())  # Convert to lowercase for case-insensitivity

    #     # Calculate the sentiment score for each word from the CSV file and aggregate
    #     csv_sentiment = sum(self.word_sentiments_csv.get(word, 0) for word in words)

    #     # Calculate the sentiment score using Sentiment Intensity Analyzer for words not found in CSV
    #     nlp_sentiment = self.sia.polarity_scores(sentence)['compound']

    #     # Combine both sentiment scores
    #     combined_sentiment = csv_sentiment + nlp_sentiment

    #     # Classify the overall sentiment based on the combined score
    #     if combined_sentiment > 0:
    #         return 'Positive'
    #     elif combined_sentiment < 0:
    #         return 'Negative'
    #     else:
    #         return 'Neutral'
        
    def write_to_txt(self, jsonPayload):
        file = r'D:\Nexensus_Projects\News\news_data_push.txt'
        with open(file, 'a+',encoding='utf-8') as f:
            f.write(json.dumps(jsonPayload))
            f.write(",")
            f.write("\n")
            f.close()
        
    def spacy_large_ner(self, document):
        named_entities_set = {(ent.text.strip(), ent.label_) for ent in self.NER(document).ents}
        return [entity[0] for entity in named_entities_set if entity[1] == 'ORG' or entity[1] == 'PRODUCT' or entity[1] == 'PERSON']
        #return {(ent.text.strip(), ent.label_) for ent in NER(document).ents}
        
        
    def match_from_starting_word(self, str1, str2):
        # Split the strings into words
        words1 = str1.split()
        words2 = str2.split()

        # Get the minimum length of the two lists of words
        min_length = min(len(words1), len(words2))

        # Iterate through the words and check if they match
        for i in range(min_length):
            if words1[i] != words2[i]:
                return False

        return True
        
        
    def getcinfromlocal(self, input_word):
        df = pd.read_csv(r"D:\Nexensus_Projects\News\active_cin_name_ac_pan.csv")
        cins = df['cin']
        company_names = df['company_name']
        cin_list = []
        for i, company_name in enumerate(company_names):
            if input_word.title() in company_name:
                matchresult = self.match_from_starting_word(input_word.title(), company_name)
                if matchresult:
                    cin_list.append(cins[i])
        return cin_list


    def load_company_info(self, input_word):
        symbols = comp_cins['Symbols']
        cins = comp_cins['CIN']
        subsidory_comps = comp_cins['all_subsidory_company']
        for index, symbol in enumerate(symbols):
            try:
                if input_word.title() in symbol.title() and len(input_word) == len(symbol.title()):
                    oneCin = cins[index]
                    subs = subsidory_comps[index]
                    all_comp = [item.strip() for item in subs.split(',')]
                                
                    return all_comp
            except Exception as e:
                return []
        return []  # Return an empty list if no matching company info is found

            
    def get_category_tag(self, sentence):
        define_tags = (self.LABOUR_NEWS_TAG + self.RAID_NEWS_TAG + self.RESIGNATION_NEWS_TAG + self.FRAUD_REGULATORY_ACTION_NEWS_TAG +
                self.MARKET_STOCK_MOVEMENT_NEWS_TAG + self.LITIGATION_NEWS_TAG + self.WHISTLEBLOWER_NEWS_TAG + self.ACCIDENT_NEWS_TAG)

        # Tokenize the sentence into words
        tokens = word_tokenize(sentence)

        # Perform part-of-speech tagging
        tagged_words = pos_tag(tokens)

        # Extract verbs, adjectives, and adverbs based on their POS tags
        verbs = [word for word, pos in tagged_words if pos.startswith('V')]
        adjectives = [word for word, pos in tagged_words if pos.startswith('J')]
        adverbs = [word for word, pos in tagged_words if pos.startswith('R')]
        nouns = [word for word, pos in tagged_words if pos.startswith('N')]
        all_tags = (verbs+adjectives+adverbs+nouns)
        filtered_tags = [tag.title() for tag in all_tags if tag.title() in define_tags or tag in define_tags]
        if len(filtered_tags)>0:
            return filtered_tags[0]
        return 'Other'
    
    def get_type_tag(self, category):
        with open(r'D:\Nexensus_Projects\News\news_type_tag.csv', mode='r') as file:
            reader = csv.DictReader(file)
            for row in reader:
                if row['Category'] == category:
                    return row['type_tag'].title()
            return "Other"
    
    def fetch_files(self, newsFiles):
        try:
            newsFile = open(newsFiles, 'r', encoding="utf8")
            for news in newsFile:
                news_payload = json.loads(news.replace('},','}').replace('\u20b9','₹').replace('\u00a0',' ').replace('\xa0',' '))
                news_heading = news_payload['heading']
                news_body = news_payload['newsBody']
                if news_heading is not None:
                    category = self.get_category_tag(news_heading)
                    sentiment = 'positive'
                    # sentiment = self.get_sentence_sentiment(news_heading)
                    type_tag = self.get_type_tag(category)
                    comp_datas = self.spacy_large_ner(news_heading)
                    if not comp_datas:
                        comp_datas = self.spacy_large_ner(' '.join(news_body.split()[:51]))
                    if comp_datas:
                        for comp_data in comp_datas:
                            compLists = self.load_company_info(comp_data.replace("'s","").replace('\u20b9','₹').replace('\u00a0',' ').replace('\xa0',' '))
                            if not compLists and len(comp_data) > 10:
                                time.sleep(1.5)
                                compLists = self.getcinfromlocal(comp_data.replace("'s","").replace('\u20b9','₹').replace('\u00a0',' ').replace('\xa0',' ').replace('&', 'and'))
                                self.news_payload = []
                                for complist in compLists:
                                        # Append a dictionary to news_payload list without reinitializing it
                                    self.news_payload.append({
                                        "companyName": None,
                                        "cin": complist,
                                        "heading": news_heading,
                                        "typeTag": type_tag,
                                        "sentiment": sentiment,
                                        "category": category,
                                        "newsDate": news_payload['newsDate'],
                                        "link": news_payload['link']
                                    })
                                if len(self.news_payload)>0:
                                    # print(json.dumps(self.news_payload))
                                    self.write_to_txt(self.news_payload)
                                    # push_news(self.news_payload)
                                    time.sleep(5)
        except Exception as e:
            print(f"Error processing news files: {str(e)}")

In [10]:
import os
import time
import nltk



file_paths = r"D:\Nexensus_Projects\News\NewsData\NewsPayload\test"
previous_files = set()

while True:
    dir_list = set(os.listdir(file_paths))
    new_files = dir_list - previous_files

    # Check if there are any new files
    if new_files:
        for file in new_files: 
            print(file)
            newsFiles = os.path.join(file_paths, file)
            NewsData().fetch_files(newsFiles)

        break                 
    else:
        pass

    # Update the set of previous files for the next iteration
    previous_files = dir_list

    # Wait for 10 minutes
    # time.sleep(7200)


data_CBI.txt


KeyboardInterrupt: 

In [5]:
# class NewsData():
#     global comp_cins, csv_file_path
    
    
#     LABOUR_NEWS_TAG = ["Labour Strikes", "Strikes", "Quits", "unrest",
#                        "agitation", "lock out", "termination", "ESI", "unrest", "Fired", "Quit", "suicide", "protest", "injury",
#                        "injured", "salary", "EPF", "attrition", "Dispute", "harassment", "protests", "Strike"]

#     RAID_NEWS_TAG = ["Raid", "Raids", "Raided"]

#     RESIGNATION_NEWS_TAG = ["Resign", "Resigns", "Resigned",
#                             "Resignation", "resigning", "management changes", "terminate"]

#     FRAUD_REGULATORY_ACTION_NEWS_TAG = ["Fraud", "Embezzlement",
#                                          "Money laundering", "Syphoning", "CBI inquiry", "Bribery", "Political Influence", "SEBI proceeding",
#                                          "regulatory action", "Penalties", "Fines", "Penalty", "Cheating", "NPA", "Default", "Criminal", "Breach",
#                                          "Scam", "Arrest", "Warrant Issued", "Probe", "Fined", "Misappropriation", "Inspection", "Complaint against",
#                                          "insolvency", "I T notice", "Hawala", "CBI", "FIR", "Jail", "Illeagal", "Forging", "booked", "scrutinize",
#                                          "land grab", "RBI", "CBI", "SEBI", "IRDA", "EXIM", "Fraudulent", "theft", "theif", "bankruptcy",
#                                          "Black Money", "Bribe", "Central Bureau of Investigation", "charge sheet", "chargesheet", "cheat",
#                                          "corrupt", "corruption", "custody", "Due", "Dues", "evading", "Fine", "Fined", "forgery", "illegal",
#                                          "Inspect", "Inspector", "investigation", "Labour Dispute", "land grab", "non compliance", "notices",
#                                          "Offered Money", "Penalized", "police", "Serious Fraud", "SFIO", "violation", "Whistleblower"]

#     MARKET_STOCK_MOVEMENT_NEWS_TAG = ["plunges", "Slowdown", "Delay",
#                                        "Delayed", "downgrade", "downgraded", "losses", "loss", "SEBI", "profit decline", "Wind Up", "plummets",
#                                        "Decline", "exits", "low", "pledge", "plummet", "SELL Recommendation", "stake sale", "stress", "suspends",
#                                        "worst", "sell", "fall", "down", "drop", "falls", "sells"]

#     LITIGATION_NEWS_TAG = ["Case Registered", "Court Denies",
#                            "Court Rejects", "contempt", "Registered Case", "Settlement", "allegations", "High Court", "District Court",
#                            "Lower Court", "Supreme Court", "Judgement", "copyrights", "defamation", "DRT", "NCLT"]

#     WHISTLEBLOWER_NEWS_TAG = ["Whistleblower"]

#     ACCIDENT_NEWS_TAG = ["Accident", "Fire", "Death", "Died",
#                          "Passed Away", "Die", "casualty", "mishap", "Blast", "casualties", "Died", "killed", "kills", "passes away",
#                          "shut down"]


#     # Combine all lists into a single list
#     all_tags = (LABOUR_NEWS_TAG + RAID_NEWS_TAG + RESIGNATION_NEWS_TAG + FRAUD_REGULATORY_ACTION_NEWS_TAG +
#                 MARKET_STOCK_MOVEMENT_NEWS_TAG + LITIGATION_NEWS_TAG + WHISTLEBLOWER_NEWS_TAG + ACCIDENT_NEWS_TAG)

#     # Print the combined list
#     #print(all_tags)

#     # load cins of companies
#     comp_cins = pd.read_excel(r'I:\NewsData\company_cin_symbol.xlsx')
#     # Load News model
#     NER = spacy.load("en_core_web_lg")
#     # Load CSV file into a DataFrame
#     csv_file_path = r'I:\NewsData\SentimentalWords.csv'  # Replace with the actual path
#     df_csv = pd.read_csv(csv_file_path)

#     # Map sentiment labels to numerical values
#     sentiment_mapping = {'Negative': -1, 'Neutral': 0, 'Positive': 1}

#     # Create a dictionary mapping words to sentiments from CSV
#     word_sentiments_csv = dict(zip(df_csv['text'], df_csv['sentiment'].map(sentiment_mapping)))

#     # Initialize the Sentiment Intensity Analyzer
#     sia = SentimentIntensityAnalyzer()

#     def get_sentence_sentiment(self, sentence):
#         # Tokenize the sentence
#         words = word_tokenize(sentence.lower())  # Convert to lowercase for case-insensitivity

#         # Calculate the sentiment score for each word from the CSV file and aggregate
#         csv_sentiment = sum(self.word_sentiments_csv.get(word, 0) for word in words)

#         # Calculate the sentiment score using Sentiment Intensity Analyzer for words not found in CSV
#         nlp_sentiment = self.sia.polarity_scores(sentence)['compound']

#         # Combine both sentiment scores
#         combined_sentiment = csv_sentiment + nlp_sentiment

#         # Classify the overall sentiment based on the combined score
#         if combined_sentiment > 0:
#             return 'Positive'
#         elif combined_sentiment < 0:
#             return 'Negative'
#         else:
#             return 'Neutral'
        
#     def write_to_txt(self, jsonPayload):
#         file = r'I:\NewsData\news_data_push_march.txt'
#         with open(file, 'a+',encoding='utf-8') as f:
#             f.write(json.dumps(jsonPayload))
#             f.write(",")
#             f.write("\n")
#             f.close()
        
#     def spacy_large_ner(self, document):
#         named_entities_set = {(ent.text.strip(), ent.label_) for ent in self.NER(document).ents}
#         return [entity[0] for entity in named_entities_set if entity[1] == 'ORG' or entity[1] == 'PRODUCT']
#         #return {(ent.text.strip(), ent.label_) for ent in NER(document).ents}
        
#     def load_company_info(self, input_word):
#         symbols = comp_cins['Symbols']
#         cins = comp_cins['CIN']
#         subsidory_comps = comp_cins['all_subsidory_company']
#         for index, symbol in enumerate(symbols):
#             try:
#                 if input_word.title() in symbol.title() and len(input_word) == len(symbol.title()):
#                     oneCin = cins[index]
#                     subs = subsidory_comps[index]
#                     all_comp = [item.strip() for item in subs.split(',')]
#                     return all_comp
#             except Exception as e:
#                 return []
#         return []  # Return an empty list if no matching company info is found

            
#     def get_type_tag(self, sentence):
#         define_tags = (self.LABOUR_NEWS_TAG + self.RAID_NEWS_TAG + self.RESIGNATION_NEWS_TAG + self.FRAUD_REGULATORY_ACTION_NEWS_TAG +
#                 self.MARKET_STOCK_MOVEMENT_NEWS_TAG + self.LITIGATION_NEWS_TAG + self.WHISTLEBLOWER_NEWS_TAG + self.ACCIDENT_NEWS_TAG)

#         # Tokenize the sentence into words
#         tokens = word_tokenize(sentence)

#         # Perform part-of-speech tagging
#         tagged_words = pos_tag(tokens)

#         # Extract verbs, adjectives, and adverbs based on their POS tags
#         verbs = [word for word, pos in tagged_words if pos.startswith('V')]
#         adjectives = [word for word, pos in tagged_words if pos.startswith('J')]
#         adverbs = [word for word, pos in tagged_words if pos.startswith('R')]
#         nouns = [word for word, pos in tagged_words if pos.startswith('N')]
#         all_tags = (verbs+adjectives+adverbs+nouns)
#         filtered_tags = [tag.title() for tag in all_tags if tag.title() in define_tags or tag in define_tags]
#         if len(filtered_tags)>0:
#             return filtered_tags[0]
#         return 'Other'
    
#     def fetch_files(self, newsFiles):
# #         folder_path = "I:/NewsData/NewsPayload/"
# #         files = os.listdir(folder_path)

# #         for file in files:
# #             newsFiles = folder_path + file
#         newsFile = open(newsFiles, 'r')
#         print("File_Name :", file)
#         for news in newsFile:
#             news_payload = json.loads(news.replace('},','}'))
#             news_heading = news_payload['heading']
#             news_body = news_payload['newsBody']
#             try:
#                 if news_heading is not None:
#                     #print("Headline ", news_heading)
#                     type_tag = self.get_type_tag(news_heading)
#                     #print(type_tag)
#                     sentiment = self.get_sentence_sentiment(news_heading)
#                     comp_datas = self.spacy_large_ner(news_heading)
#                     if len(comp_datas)>0:
#                         for comp_data in comp_datas:
#                             comp_data = comp_data.replace("'s", "")
#                             compLists = self.load_company_info(comp_data)
#                             #print("$$$ ", compLists)
#                             if compLists is not None and len(compLists) > 0:
#                                 self.news_payload = []
#                                 for complist in compLists:
#                                     # Append a dictionary to news_payload list without reinitializing it
#                                     self.news_payload.append({
#                                         "companyName": None,
#                                         "cin": complist,
#                                         "heading": news_heading,
#                                         "typeTag": type_tag,
#                                         "sentiment": sentiment,
#                                         "newsDate": news_payload['newsDate'],
#                                         "link": news_payload['link']
#                                     })
#                                 # print(json.dumps(self.news_payload))
#                                 self.write_to_txt(self.news_payload)
#                                 push_news(self.news_payload)
#                                 time.sleep(5)
#             except Exception as e:
#                 print("Exception :", e)

In [ ]:


# import requests
# import json

# def push_news(news_payload):
#     newsUrl = "103.233.79.196:8083/background_scheduler/news/processNews"
#     header = {
#         "Content-Type": "application/json"
#     }
    
#     resp = requests.post(url = newsUrl, data = json.dumps(news_payload), headers = header)
#     return resp.content

In [ ]:
# # import pandas as pd
# # import spacy
# # from nltk.tokenize import word_tokenize
# # from nltk import pos_tag
# # from nltk.sentiment.vader import SentimentIntensityAnalyzer
# # import json
# # import os

# class NewsData():
#     global comp_cins, csv_file_path
    
    
#     LABOUR_NEWS_TAG = ["Labour Strikes", "Strikes", "Quits", "unrest",
#                        "agitation", "lock out", "termination", "ESI", "unrest", "Fired", "Quit", "suicide", "protest", "injury",
#                        "injured", "salary", "EPF", "attrition", "Dispute", "harassment", "protests", "Strike"]

#     RAID_NEWS_TAG = ["Raid", "Raids", "Raided"]

#     RESIGNATION_NEWS_TAG = ["Resign", "Resigns", "Resigned",
#                             "Resignation", "resigning", "management changes", "terminate"]

#     FRAUD_REGULATORY_ACTION_NEWS_TAG = ["Fraud", "Embezzlement",
#                                          "Money laundering", "Syphoning", "CBI inquiry", "Bribery", "Political Influence", "SEBI proceeding",
#                                          "regulatory action", "Penalties", "Fines", "Penalty", "Cheating", "NPA", "Default", "Criminal", "Breach",
#                                          "Scam", "Arrest", "Warrant Issued", "Probe", "Fined", "Misappropriation", "Inspection", "Complaint against",
#                                          "insolvency", "I T notice", "Hawala", "CBI", "FIR", "Jail", "Illeagal", "Forging", "booked", "scrutinize",
#                                          "land grab", "RBI", "CBI", "SEBI", "IRDA", "EXIM", "Fraudulent", "theft", "theif", "bankruptcy",
#                                          "Black Money", "Bribe", "Central Bureau of Investigation", "charge sheet", "chargesheet", "cheat",
#                                          "corrupt", "corruption", "custody", "Due", "Dues", "evading", "Fine", "Fined", "forgery", "illegal",
#                                          "Inspect", "Inspector", "investigation", "Labour Dispute", "land grab", "non compliance", "notices",
#                                          "Offered Money", "Penalized", "police", "Serious Fraud", "SFIO", "violation", "Whistleblower"]

#     MARKET_STOCK_MOVEMENT_NEWS_TAG = ["plunges", "Slowdown", "Delay",
#                                        "Delayed", "downgrade", "downgraded", "losses", "loss", "SEBI", "profit decline", "Wind Up", "plummets",
#                                        "Decline", "exits", "low", "pledge", "plummet", "SELL Recommendation", "stake sale", "stress", "suspends",
#                                        "worst", "sell", "fall", "down", "drop", "falls", "sells"]

#     LITIGATION_NEWS_TAG = ["Case Registered", "Court Denies",
#                            "Court Rejects", "contempt", "Registered Case", "Settlement", "allegations", "High Court", "District Court",
#                            "Lower Court", "Supreme Court", "Judgement", "copyrights", "defamation", "DRT", "NCLT"]

#     WHISTLEBLOWER_NEWS_TAG = ["Whistleblower"]

#     ACCIDENT_NEWS_TAG = ["Accident", "Fire", "Death", "Died",
#                          "Passed Away", "Die", "casualty", "mishap", "Blast", "casualties", "Died", "killed", "kills", "passes away",
#                          "shut down"]


#     # Combine all lists into a single list
#     all_tags = (LABOUR_NEWS_TAG + RAID_NEWS_TAG + RESIGNATION_NEWS_TAG + FRAUD_REGULATORY_ACTION_NEWS_TAG +
#                 MARKET_STOCK_MOVEMENT_NEWS_TAG + LITIGATION_NEWS_TAG + WHISTLEBLOWER_NEWS_TAG + ACCIDENT_NEWS_TAG)

#     # Print the combined list
#     #print(all_tags)

#     # load cins of companies
#     comp_cins = pd.read_excel(r'D:\Shivam\NewsCins\company_cin_symbol.xlsx')
#     # Load News model
#     NER = spacy.load("en_core_web_lg")
#     # Load CSV file into a DataFrame
#     csv_file_path = r'D:\ExcelFiles\SentimentalWords.csv'  # Replace with the actual path
#     df_csv = pd.read_csv(csv_file_path)

#     # Map sentiment labels to numerical values
#     sentiment_mapping = {'Negative': -1, 'Neutral': 0, 'Positive': 1}

#     # Create a dictionary mapping words to sentiments from CSV
#     word_sentiments_csv = dict(zip(df_csv['text'], df_csv['sentiment'].map(sentiment_mapping)))

#     # Initialize the Sentiment Intensity Analyzer
#     sia = SentimentIntensityAnalyzer()

#     def get_sentence_sentiment(self, sentence):
#         # Tokenize the sentence
#         words = word_tokenize(sentence.lower())  # Convert to lowercase for case-insensitivity

#         # Calculate the sentiment score for each word from the CSV file and aggregate
#         csv_sentiment = sum(self.word_sentiments_csv.get(word, 0) for word in words)

#         # Calculate the sentiment score using Sentiment Intensity Analyzer for words not found in CSV
#         nlp_sentiment = self.sia.polarity_scores(sentence)['compound']

#         # Combine both sentiment scores
#         combined_sentiment = csv_sentiment + nlp_sentiment

#         # Classify the overall sentiment based on the combined score
#         if combined_sentiment > 0:
#             return 'positive'
#         elif combined_sentiment < 0:
#             return 'negative'
#         else:
#             return 'neutral'
        
#     def write_to_txt(self, jsonPayload):
#         file = r'D:\News_payload\news_data.txt'
#         with open(file, 'a+',encoding='utf-8') as f:
#             f.write(json.dumps(jsonPayload))
#             f.write(",")
#             f.write("\n")
#             f.close()
        
#     def spacy_large_ner(self, document):
#         named_entities_set = {(ent.text.strip(), ent.label_) for ent in self.NER(document).ents}
#         return [entity[0] for entity in named_entities_set if entity[1] == 'ORG' or entity[1] == 'PRODUCT']
#         #return {(ent.text.strip(), ent.label_) for ent in NER(document).ents}
        
#     def load_company_info(self, input_word):
#         symbols = comp_cins['Symbols']
#         cins = comp_cins['CIN']
#         subsidory_comps = comp_cins['all_subsidory_company']
#         for index, symbol in enumerate(symbols):
#             try:
#                 if input_word.title() in symbol.title() and len(input_word) == len(symbol.title()):
#                     oneCin = cins[index]
#                     subs = subsidory_comps[index]
#                     all_comp = [item.strip() for item in subs.split(',')]
#                     return all_comp
#             except Exception as e:
#                 return []
#         return []  # Return an empty list if no matching company info is found

            
#     def get_type_tag(self, sentence):
#         define_tags = (self.LABOUR_NEWS_TAG + self.RAID_NEWS_TAG + self.RESIGNATION_NEWS_TAG + self.FRAUD_REGULATORY_ACTION_NEWS_TAG +
#                 self.MARKET_STOCK_MOVEMENT_NEWS_TAG + self.LITIGATION_NEWS_TAG + self.WHISTLEBLOWER_NEWS_TAG + self.ACCIDENT_NEWS_TAG)

#         # Tokenize the sentence into words
#         tokens = word_tokenize(sentence)

#         # Perform part-of-speech tagging
#         tagged_words = pos_tag(tokens)

#         # Extract verbs, adjectives, and adverbs based on their POS tags
#         verbs = [word for word, pos in tagged_words if pos.startswith('V')]
#         adjectives = [word for word, pos in tagged_words if pos.startswith('J')]
#         adverbs = [word for word, pos in tagged_words if pos.startswith('R')]
#         nouns = [word for word, pos in tagged_words if pos.startswith('N')]
#         all_tags = (verbs+adjectives+adverbs+nouns)
#         filtered_tags = [tag.title() for tag in all_tags if tag.title() in define_tags or tag in define_tags]
#         if len(filtered_tags)>0:
#             return filtered_tags[0]
#         return 'Other'
    
#     def fetch_files(self):
#         folder_path = "D:/TextFiles/"
#         files = os.listdir(folder_path)

#         for file in files:
#             newsFiles = folder_path + file
#             newsFile = open(newsFiles, 'r')
#             for news in newsFile:
#                 news_payload = json.loads(news.replace('},','}'))
#                 news_heading = news_payload['heading']
#                 news_body = news_payload['newsBody']
#                 if news_heading is not None:
#                     print("Headline ", news_heading)
#                     type_tag = self.get_type_tag(news_heading)
#                     #print(type_tag)
#                     sentiment = self.get_sentence_sentiment(news_heading)
#                     comp_datas = self.spacy_large_ner(news_heading)
#                     if len(comp_datas)>0:
#                         for comp_data in comp_datas:
#                             compLists = self.load_company_info(comp_data)
#                             #print("$$$ ", compLists)
#                             if compLists is not None and len(compLists) > 0:
#                                 self.news_payload = []
#                                 for complist in compLists:
#                                     # Append a dictionary to news_payload list without reinitializing it
#                                     self.news_payload.append({
#                                         "companyName": None,
#                                         "cin": complist,
#                                         "heading": news_heading,
#                                         "typeTag": type_tag,
#                                         "sentiment": sentiment,
#                                         "newsDate": news_payload['newsDate'],
#                                         "link": news_payload['link']
#                                     })
#                                 # print(json.dumps(self.news_payload))
#                                 self.write_to_txt(self.news_payload)
#                                 # push_news(self.news_payload)

In [ ]:
# news_data = NewsData()
# news_data.fetch_files()

In [ ]:
# import pandas as pd
# import spacy
# from nltk.tokenize import word_tokenize
# from nltk import pos_tag
# from nltk.sentiment.vader import SentimentIntensityAnalyzer
# import json
# import os

# class NewsData():
#     def __init__(self, comp_cins_path, csv_file_path):
#         self.comp_cins = pd.read_excel(comp_cins_path)
#         self.ner_model = spacy.load("en_core_web_lg")
#         self.df_csv = pd.read_csv(csv_file_path)
#         self.word_sentiments = self._create_sentiment_mapping()

#     def _create_sentiment_mapping(self):
#         sentiment_mapping = {'Negative': -1, 'Neutral': 0, 'Positive': 1}
#         return dict(zip(self.df_csv['text'], self.df_csv['sentiment'].map(sentiment_mapping)))

#     def _get_sentiment(self, sentence):
#         sia = SentimentIntensityAnalyzer()
#         words = word_tokenize(sentence.lower())
#         csv_sentiment = sum(self.word_sentiments.get(word, 0) for word in words)
#         nlp_sentiment = sia.polarity_scores(sentence)['compound']
#         combined_sentiment = csv_sentiment + nlp_sentiment
#         return 'positive' if combined_sentiment > 0 else ('negative' if combined_sentiment < 0 else 'neutral')

#     def _get_named_entities(self, document):
#         named_entities = self.ner_model(document).ents
#         org_entities = [entity.text.strip() for entity in named_entities if entity.label_ in ('ORG', 'PRODUCT')]
#         return org_entities

#     def _find_company_info(self, input_word):
#         symbols = self.comp_cins['Symbols']
#         cins = self.comp_cins['CIN']
#         subsidiary_companies = self.comp_cins['all_subsidory_company']
#         for index, symbol in enumerate(symbols):
#             if input_word.title() in symbol.title() and len(input_word) == len(symbol.title()):
#                 return [subsidiary_companies[index]]
#         return []

#     def _get_news_type(self, sentence):
#         all_tags = (
#             self.LABOUR_NEWS_TAG + self.RAID_NEWS_TAG + self.RESIGNATION_NEWS_TAG + 
#             self.FRAUD_REGULATORY_ACTION_NEWS_TAG + self.MARKET_STOCK_MOVEMENT_NEWS_TAG + 
#             self.LITIGATION_NEWS_TAG + self.WHISTLEBLOWER_NEWS_TAG + self.ACCIDENT_NEWS_TAG
#         )
#         tokens = word_tokenize(sentence)
#         tagged_words = pos_tag(tokens)
#         all_tags = [word.title() for word, pos in tagged_words if pos.startswith(('V', 'J', 'R', 'N'))]
#         filtered_tags = [tag for tag in all_tags if tag in all_tags]
#         return filtered_tags[0] if filtered_tags else 'Other'

#     def fetch_files(self, folder_path):
#         files = os.listdir(folder_path)
#         for file in files:
#             news_file_path = os.path.join(folder_path, file)
#             with open(news_file_path, 'r') as news_file:
#                 for news in news_file:
#                     news_payload = json.loads(news.replace('},', '}'))
#                     news_heading = news_payload.get('heading')
#                     if news_heading:
#                         sentiment = self._get_sentiment(news_heading)
#                         type_tag = self._get_news_type(news_heading)
#                         comp_datas = self._get_named_entities(news_heading)
#                         for comp_data in comp_datas:
#                             comp_lists = self._find_company_info(comp_data)
#                             if comp_lists:
#                                 self.news_payload = [{
#                                     "companyName": None,
#                                     "cin": comp,
#                                     "heading": news_heading,
#                                     "typeTag": type_tag,
#                                     "sentiment": sentiment,
#                                     "newsDate": news_payload['newsDate'],
#                                     "link": news_payload['link']
#                                 } for comp in comp_lists]
#                                 print(self.news_payload)

# # Define class constants
# NewsData.LABOUR_NEWS_TAG = ["Labour Strikes", "Strikes", "Quits", "unrest", "agitation", "lock out", "termination", "ESI", "unrest", "Fired", "Quit", "suicide", "protest", "injury", "injured", "salary", "EPF", "attrition", "Dispute", "harassment", "protests", "Strike"]
# NewsData.RAID_NEWS_TAG = ["Raid", "Raids", "Raided"]
# NewsData.RESIGNATION_NEWS_TAG = ["Resign", "Resigns", "Resigned", "Resignation", "resigning", "management changes", "terminate"]
# NewsData.FRAUD_REGULATORY_ACTION_NEWS_TAG = ["Fraud", "Embezzlement", "Money laundering", "Syphoning", "CBI inquiry", "Bribery", "Political Influence", "SEBI proceeding", "regulatory action", "Penalties", "Fines", "Penalty", "Cheating", "NPA", "Default", "Criminal", "Breach", "Scam", "Arrest", "Warrant Issued", "Probe", "Fined", "Misappropriation", "Inspection", "Complaint against", "insolvency", "I T notice", "Hawala", "CBI", "FIR", "Jail", "Illeagal", "Forging", "booked", "scrutinize", "land grab", "RBI", "CBI", "SEBI", "IRDA", "EXIM", "Fraudulent", "theft", "theif", "bankruptcy", "Black Money", "Bribe", "Central Bureau of Investigation", "charge sheet", "chargesheet", "cheat", "corrupt", "corruption", "custody", "Due", "Dues", "evading", "Fine", "Fined", "forgery", "illegal", "Inspect", "Inspector", "investigation", "Labour Dispute", "land grab", "non compliance", "notices", "Offered Money", "Penalized", "police", "Serious Fraud", "SFIO", "violation", "Whistleblower"]
# NewsData.MARKET_STOCK_MOVEMENT_NEWS_TAG = ["plunges", "Slowdown", "Delay", "Delayed", "downgrade", "downgraded", "losses", "loss", "SEBI", "profit decline", "Wind Up", "plummets", "Decline", "exits", "low", "pledge", "plummet", "SELL Recommendation", "stake sale", "stress", "suspends", "worst", "sell", "fall", "down", "drop", "falls", "sells"]
# NewsData.LITIGATION_NEWS_TAG = ["Case Registered", "Court Denies", "Court Rejects", "contempt", "Registered Case", "Settlement", "allegations", "High Court", "District Court", "Lower Court", "Supreme Court", "Judgement", "copyrights", "defamation", "DRT", "NCLT"]
# NewsData.WHISTLEBLOWER_NEWS_TAG = ["Whistleblower"]
# NewsData.ACCIDENT_NEWS_TAG = ["Accident", "Fire", "Death", "Died", "Passed Away", "Die", "casualty", "mishap", "Blast", "casualties", "Died", "killed", "kills", "passes away", "shut down"]

# # Example Usage:
# if __name__ == "__main__":
#     news_data = NewsData(r'D:\Shivam\NewsCins\company_cin_symbol.xlsx', r'D:\ExcelFiles\SentimentalWords.csv')
#     news_data.fetch_files("D:/TextFiles/")


In [ ]:
import requests
import json

def push_news(news_payload):
    newsUrl = "http://localhost:8087/news/processNews"
    header = {
        "Content-Type": "application/json"
    }
    data = [{'companyName': None, 'cin': 'L24239MH1935PLC002380', 'heading': "Pharma Outlook 2024: How are Cipla, Sun Pharma, Dr Reddy's and others likely to perform this year?", 'typeTag': 'Other', 'sentiment': 'neutral', 'newsDate': '2024-01-01', 'link': 'https://www.livemint.com/companies/pharma-outlook-2024-how-are-cipla-sun-pharma-dr-reddys-and-others-likely-to-perform-next-year-11703768648557.html'}, {'companyName': None, 'cin': 'AAO-3042', 'heading': "Pharma Outlook 2024: How are Cipla, Sun Pharma, Dr Reddy's and others likely to perform this year?", 'typeTag': 'Other', 'sentiment': 'neutral', 'newsDate': '2024-01-01', 'link': 'https://www.livemint.com/companies/pharma-outlook-2024-how-are-cipla-sun-pharma-dr-reddys-and-others-likely-to-perform-next-year-11703768648557.html'}, {'companyName': None, 'cin': 'FOREN0000000000006111', 'heading': "Pharma Outlook 2024: How are Cipla, Sun Pharma, Dr Reddy's and others likely to perform this year?", 'typeTag': 'Other', 'sentiment': 'neutral', 'newsDate': '2024-01-01', 'link': 'https://www.livemint.com/companies/pharma-outlook-2024-how-are-cipla-sun-pharma-dr-reddys-and-others-likely-to-perform-next-year-11703768648557.html'}, {'companyName': None, 'cin': 'U73100KA2006PTC038256', 'heading': "Pharma Outlook 2024: How are Cipla, Sun Pharma, Dr Reddy's and others likely to perform this year?", 'typeTag': 'Other', 'sentiment': 'neutral', 'newsDate': '2024-01-01', 'link': 'https://www.livemint.com/companies/pharma-outlook-2024-how-are-cipla-sun-pharma-dr-reddys-and-others-likely-to-perform-next-year-11703768648557.html'}, {'companyName': None, 'cin': 'U33111MH2012PTC234037', 'heading': "Pharma Outlook 2024: How are Cipla, Sun Pharma, Dr Reddy's and others likely to perform this year?", 'typeTag': 'Other', 'sentiment': 'neutral', 'newsDate': '2024-01-01', 'link': 'https://www.livemint.com/companies/pharma-outlook-2024-how-are-cipla-sun-pharma-dr-reddys-and-others-likely-to-perform-next-year-11703768648557.html'}, {'companyName': None, 'cin': 'FOREN0000000000006113', 'heading': "Pharma Outlook 2024: How are Cipla, Sun Pharma, Dr Reddy's and others likely to perform this year?", 'typeTag': 'Other', 'sentiment': 'neutral', 'newsDate': '2024-01-01', 'link': 'https://www.livemint.com/companies/pharma-outlook-2024-how-are-cipla-sun-pharma-dr-reddys-and-others-likely-to-perform-next-year-11703768648557.html'}, {'companyName': None, 'cin': 'U74999MH2015PTC263070', 'heading': "Pharma Outlook 2024: How are Cipla, Sun Pharma, Dr Reddy's and others likely to perform this year?", 'typeTag': 'Other', 'sentiment': 'neutral', 'newsDate': '2024-01-01', 'link': 'https://www.livemint.com/companies/pharma-outlook-2024-how-are-cipla-sun-pharma-dr-reddys-and-others-likely-to-perform-next-year-11703768648557.html'}, {'companyName': None, 'cin': 'U74996DL2019PTC345639', 'heading': "Pharma Outlook 2024: How are Cipla, Sun Pharma, Dr Reddy's and others likely to perform this year?", 'typeTag': 'Other', 'sentiment': 'neutral', 'newsDate': '2024-01-01', 'link': 'https://www.livemint.com/companies/pharma-outlook-2024-how-are-cipla-sun-pharma-dr-reddys-and-others-likely-to-perform-next-year-11703768648557.html'}, {'companyName': None, 'cin': 'U24239MH2019PLC333266', 'heading': "Pharma Outlook 2024: How are Cipla, Sun Pharma, Dr Reddy's and others likely to perform this year?", 'typeTag': 'Other', 'sentiment': 'neutral', 'newsDate': '2024-01-01', 'link': 'https://www.livemint.com/companies/pharma-outlook-2024-how-are-cipla-sun-pharma-dr-reddys-and-others-likely-to-perform-next-year-11703768648557.html'}, {'companyName': None, 'cin': 'U24100MH2015PLC267880', 'heading': "Pharma Outlook 2024: How are Cipla, Sun Pharma, Dr Reddy's and others likely to perform this year?", 'typeTag': 'Other', 'sentiment': 'neutral', 'newsDate': '2024-01-01', 'link': 'https://www.livemint.com/companies/pharma-outlook-2024-how-are-cipla-sun-pharma-dr-reddys-and-others-likely-to-perform-next-year-11703768648557.html'}, {'companyName': None, 'cin': 'FOREN0000000000006112', 'heading': "Pharma Outlook 2024: How are Cipla, Sun Pharma, Dr Reddy's and others likely to perform this year?", 'typeTag': 'Other', 'sentiment': 'neutral', 'newsDate': '2024-01-01', 'link': 'https://www.livemint.com/companies/pharma-outlook-2024-how-are-cipla-sun-pharma-dr-reddys-and-others-likely-to-perform-next-year-11703768648557.html'}, {'companyName': None, 'cin': 'U72900MH2021PTC359833', 'heading': "Pharma Outlook 2024: How are Cipla, Sun Pharma, Dr Reddy's and others likely to perform this year?", 'typeTag': 'Other', 'sentiment': 'neutral', 'newsDate': '2024-01-01', 'link': 'https://www.livemint.com/companies/pharma-outlook-2024-how-are-cipla-sun-pharma-dr-reddys-and-others-likely-to-perform-next-year-11703768648557.html'}, {'companyName': None, 'cin': 'U40106DL2020PTC373516', 'heading': "Pharma Outlook 2024: How are Cipla, Sun Pharma, Dr Reddy's and others likely to perform this year?", 'typeTag': 'Other', 'sentiment': 'neutral', 'newsDate': '2024-01-01', 'link': 'https://www.livemint.com/companies/pharma-outlook-2024-how-are-cipla-sun-pharma-dr-reddys-and-others-likely-to-perform-next-year-11703768648557.html'}, {'companyName': None, 'cin': 'U72900MH2022PLC377512', 'heading': "Pharma Outlook 2024: How are Cipla, Sun Pharma, Dr Reddy's and others likely to perform this year?", 'typeTag': 'Other', 'sentiment': 'neutral', 'newsDate': '2024-01-01', 'link': 'https://www.livemint.com/companies/pharma-outlook-2024-how-are-cipla-sun-pharma-dr-reddys-and-others-likely-to-perform-next-year-11703768648557.html'}],



    resp = requests.post(url = newsUrl, data = json.dumps(news_payload), headers = header)
    return resp.content


In [ ]:
# files = open(r"D:\News_payload\news_data.txt", "r")
# for file in files:
#     news = file.replace('}],','}]')
#     newsPayload = news.replace('None','null').replace("'",'"')
#     news_payload = json.loads(newsPayload)
#     push_news(news_payload)
    

In [2]:
# Load News model
import spacy
NER_sm = spacy.load("en_core_web_sm")
NER = spacy.load("en_core_web_lg")


def spacy_large_ner(document):
    named_entities_set = {(ent.text.strip(), ent.label_) for ent in NER(document).ents}
    return [entity[0] for entity in named_entities_set if entity[1] == 'ORG' or entity[1] == 'PRODUCT' or entity[1] == 'PERSON']

In [3]:
spacy_large_ner(") G S Mahanagar Co Op Bank Limited Through Its Branch Manager")

['G S Mahanagar Co Op Bank Limited Through Its Branch']